In [26]:
import pandas as pd

# One line per file. encoding="utf-8-sig" just tells pandas to ignore
# an invisible hidden character at the start of these files.
sales = pd.read_csv("../Data/Raw/Sales_Transactions.csv", encoding="utf-8-sig")
cust  = pd.read_csv("../Data/Raw/Customers.csv", encoding="utf-8-sig")
pm    = pd.read_csv("../Data/Raw/Product_Master.csv", encoding="utf-8-sig")
qual  = pd.read_csv("../Data/Raw/Quality.csv", encoding="utf-8-sig")
prod  = pd.read_csv("../Data/Raw/Production_Orders.csv", encoding="utf-8-sig")
bom   = pd.read_csv("../Data/Raw/Product_BOM.csv", encoding="utf-8-sig")
po    = pd.read_csv("../Data/Raw/Purchase_Orders.csv", encoding="utf-8-sig")
sup   = pd.read_csv("../Data/Raw/Suppliers.csv", encoding="utf-8-sig")
inv   = pd.read_csv("../Data/Raw/Inventory_History.csv", encoding="utf-8-sig")

# Prove they opened correctly
print("sales rows:", len(sales))
print("customers rows:", len(cust))
print("products rows:", len(pm))
print("quality rows:", len(qual))
print("production rows:", len(prod))
print("bom rows:", len(bom))
print("purchase orders rows:", len(po))
print("suppliers rows:", len(sup))
print("inventory rows:", len(inv))

sales rows: 180000
customers rows: 30000
products rows: 59
quality rows: 9000
production rows: 2500
bom rows: 337
purchase orders rows: 3600
suppliers rows: 12
inventory rows: 6372


In [27]:
# CHECK 1: every customer in Sales must exist in Customers
bad = sales[~sales["Customer_ID"].isin(cust["Customer_ID"])]
print("Sales rows with UNKNOWN Customer_ID:", len(bad))

# CHECK 2: every product in Sales must exist in Product_Master
bad = sales[~sales["Product_ID"].isin(pm["Product_ID"])]
print("Sales rows with UNKNOWN Product_ID:", len(bad))

# CHECK 3: every product in Quality must exist in Product_Master
bad = qual[~qual["Product_ID"].isin(pm["Product_ID"])]
print("Quality rows with UNKNOWN Product_ID:", len(bad))

# CHECK 4: every product in Production must exist in Product_Master
bad = prod[~prod["Product_ID"].isin(pm["Product_ID"])]
print("Production rows with UNKNOWN Product_ID:", len(bad))

# CHECK 5: every product in BOM must exist in Product_Master
bad = bom[~bom["Product_ID"].isin(pm["Product_ID"])]
print("BOM rows with UNKNOWN Product_ID:", len(bad))

# CHECK 6: every material in BOM must appear in Purchase_Orders
bad = bom[~bom["Material_ID"].isin(po["Material_ID"])]
print("BOM rows with NEVER-BOUGHT Material_ID:", len(bad))

# CHECK 7: every supplier in Purchase_Orders must exist in Suppliers
bad = po[~po["Supplier_ID"].isin(sup["Supplier_ID"])]
print("PO rows with UNKNOWN Supplier_ID:", len(bad))

# CHECK 8: every product in Inventory must exist in Product_Master
bad = inv[~inv["Product_ID"].isin(pm["Product_ID"])]
print("Inventory rows with UNKNOWN Product_ID:", len(bad))

Sales rows with UNKNOWN Customer_ID: 0
Sales rows with UNKNOWN Product_ID: 0
Quality rows with UNKNOWN Product_ID: 0
Production rows with UNKNOWN Product_ID: 0
BOM rows with UNKNOWN Product_ID: 0
BOM rows with NEVER-BOUGHT Material_ID: 0
PO rows with UNKNOWN Supplier_ID: 0
Inventory rows with UNKNOWN Product_ID: 0


In [37]:
sales.head()

,Row_ID,Invoice_ID,Order_Date,Month_Key,Location_ID,Location_Name,Channel,Customer_ID,Customer_Type,Customer_Governorate,...,Estimated_COGS_EGP,Gross_Profit_EGP,Gross_Margin_Pct,Payment_Method,Returned,Return_Date,Return_Reason,Refund_Value_EGP,Quality_Related_Return,Data_Class
0,0,INV0000001,2024-05-11,2024-05,LOC002,LEO MODA Online Store,Online,C18976,Retail,Cairo,...,247.57,82.23,0.2493,Wallet,No,NaN,No return,0.0,No,Modeled training transaction anchored to publi...
1,1,INV0000002,2025-07-31,2025-07,LOC001,LEO MODA - Sidi Bishr Store,Retail,C22543,Retail,Alexandria,...,163.31,67.89,0.2936,Card,No,NaN,No return,0.0,No,Modeled training transaction anchored to publi...
2,2,INV0000003,2026-05-23,2026-05,LOC003,El-Sharnouby Wholesale Desk,Wholesale,C13351,Wholesale,Alexandria,...,4301.71,512.69,0.1065,Cash,No,NaN,No return,0.0,No,Modeled training transaction anchored to publi...
3,3,INV0000004,2024-06-05,2024-06,LOC003,El-Sharnouby Wholesale Desk,Wholesale,C21377,Wholesale,Dakahlia,...,10451.13,907.59,0.0799,Bank Transfer,No,NaN,No return,0.0,No,Modeled training transaction anchored to publi...
4,4,INV0000005,2025-12-28,2025-12,LOC001,LEO MODA - Sidi Bishr Store,Retail,C21499,Retail,Alexandria,...,186.16,53.04,0.2217,Cash,No,NaN,No return,0.0,No,Modeled training transaction anchored to publi...


In [28]:
# First LOOK at what is inside this column (True/False? or text?)
print(sales["Quality_Related_Return"].value_counts())

Quality_Related_Return
No     177543
Yes      2457
Name: count, dtype: int64


In [29]:
q = sales[sales["Quality_Related_Return"] == "Yes"]

In [48]:
print("Number of quality-related returns:", len(q))
print("Total refund money (EGP):", q["Refund_Value_EGP"].sum())

# Which product families cause the most broken returns?
print(q["Product_Family"].value_counts())

Number of quality-related returns: 2457
Total refund money (EGP): 2509899.21
Product_Family
Pajamas/Homewear    1753
Sweatshirts          286
T-Shirts             225
Kids Sets            193
Name: count, dtype: int64


In [31]:
# 1. Filter the sales table to ONLY look at broken pajamas
bad_pajamas = sales[
    (sales["Quality_Related_Return"] == "Yes") & 
    (sales["Product_Family"] == "Pajamas/Homewear")
]

print("Number of broken pajama sales:", len(bad_pajamas))
print("Money lost on broken pajamas:", bad_pajamas["Refund_Value_EGP"].sum())

Number of broken pajama sales: 1753
Money lost on broken pajamas: 1608861.16


In [32]:
# 2. Look up the factory defects for these specific pajama products
pajama_defects = pd.merge(
    bad_pajamas[["Product_ID", "Product_Name", "Refund_Value_EGP"]], 
    qual[["Product_ID", "Defect_Type", "Root_Cause", "Estimated_Loss_EGP"]], 
    on="Product_ID", 
    how="left"
)

print("\n--- TOP REASONS PAJAMAS ARE BREAKING (Root Cause) ---")
print(pajama_defects["Root_Cause"].value_counts().head())

print("\n--- TOP TYPES OF DEFECTS ---")
print(pajama_defects["Defect_Type"].value_counts().head())


--- TOP REASONS PAJAMAS ARE BREAKING (Root Cause) ---
Root_Cause
No significant issue     73476
Handling / packing       39359
Curing temperature       38399
Pattern / measurement    37998
Process variation        37482
Name: count, dtype: int64

--- TOP TYPES OF DEFECTS ---
Defect_Type
Size Variance         36497
Stitch Defect         34790
Ink Smudge            33348
Packing Error         33311
Color Misalignment    33110
Name: count, dtype: int64


In [33]:
# 3. Look up the production orders for these pajamas
pajama_production = pd.merge(
    bad_pajamas[["Product_ID"]].drop_duplicates(), # drop_duplicates means we only check each product once
    prod[["Product_ID", "Production_Order_ID", "Line_ID", "Rejected_Qty", "Primary_Bottleneck"]], 
    on="Product_ID", 
    how="left"
)

print("\n--- FACTORY LINES MAKING PAJAMAS ---")
print(pajama_production["Line_ID"].value_counts())

print("\n--- FACTORY BOTTLENECKS ---")
print(pajama_production["Primary_Bottleneck"].value_counts().head())


--- FACTORY LINES MAKING PAJAMAS ---
Line_ID
LINE-01    367
LINE-04    364
LINE-05    347
LINE-02    347
LINE-03    336
Name: count, dtype: int64

--- FACTORY BOTTLENECKS ---
Primary_Bottleneck
Sewing       95
Finishing    80
Cutting      45
Packing      39
Printing     33
Name: count, dtype: int64


In [34]:
# 4a. What materials are inside the pajamas?
pajama_materials = pd.merge(
    bad_pajamas[["Product_ID"]].drop_duplicates(),
    bom[["Product_ID", "Material_ID", "Material_Name"]],
    on="Product_ID",
    how="left"
)

# 4b. Who supplied those materials, and did they arrive late or get rejected?
material_suppliers = pd.merge(
    pajama_materials,
    po[["Material_ID", "Supplier_ID", "Rejected_Qty", "Promised_Date", "Actual_Receipt_Date"]],
    on="Material_ID",
    how="left"
)

# 4c. Get the actual names of the suppliers
final_suspects = pd.merge(
    material_suppliers,
    sup[["Supplier_ID", "Supplier_Name"]],
    on="Supplier_ID",
    how="left"
)

print("\n--- SUSPECT SUPPLIERS FOR PAJAMA MATERIALS ---")
print(final_suspects["Supplier_Name"].value_counts())

# Let's see which supplier had the most rejected materials!
print("\n--- SUPPLIERS WITH THE MOST REJECTED MATERIALS ---")
print(final_suspects.groupby("Supplier_Name")["Rejected_Qty"].sum().sort_values(ascending=False).head())


--- SUSPECT SUPPLIERS FOR PAJAMA MATERIALS ---
Supplier_Name
Approved Supplier 07 (anonymized)    6603
Approved Supplier 08 (anonymized)    6382
Approved Supplier 06 (anonymized)    6377
Approved Supplier 11 (anonymized)    6039
Approved Supplier 05 (anonymized)    6036
Approved Supplier 09 (anonymized)    6021
Approved Supplier 12 (anonymized)    5895
Approved Supplier 03 (anonymized)    5884
Approved Supplier 02 (anonymized)    5841
Approved Supplier 04 (anonymized)    5832
Approved Supplier 10 (anonymized)    5829
Approved Supplier 01 (anonymized)    5827
Name: count, dtype: int64

--- SUPPLIERS WITH THE MOST REJECTED MATERIALS ---
Supplier_Name
Approved Supplier 11 (anonymized)    742764
Approved Supplier 09 (anonymized)    664819
Approved Supplier 06 (anonymized)    661235
Approved Supplier 07 (anonymized)    619993
Approved Supplier 05 (anonymized)    614771
Name: Rejected_Qty, dtype: int64


In [35]:
# 1. Calculate total money lost to ALL broken products (Refunds)
total_refund_loss = sales[sales["Quality_Related_Return"] == "Yes"]["Refund_Value_EGP"].sum()

# 2. Calculate total money lost inside the factory (Scrap/Rework from Quality.csv)
# Estimated_Loss_EGP is the money the factory threw in the trash because of defects
total_factory_loss = qual["Estimated_Loss_EGP"].sum()

# 3. Calculate total money lost from Printing mistakes (Printing_Batches.csv)
# We need to load the printing file first
printing = pd.read_csv("../Data/Raw/Printing_Batches.csv", encoding="utf-8-sig")
total_printing_loss = printing["Rework_Cost_EGP"].sum() + printing["Ink_or_Consumable_Cost_EGP"].sum()

# 4. The Grand Total
grand_total_loss = total_refund_loss + total_factory_loss + total_printing_loss

print("--- THE TRUE COST OF POOR QUALITY (CPQ) ---")
print(f"1. Customer Refunds (Broken items sent back): {total_refund_loss:,.0f} EGP")
print(f"2. Factory Scrap (Items thrown in the trash): {total_factory_loss:,.0f} EGP")
print(f"3. Printing Rework (Ink and fixing mistakes): {total_printing_loss:,.0f} EGP")
print("-" * 45)
print(f"TOTAL MONEY BLEEDING: {grand_total_loss:,.0f} EGP")

--- THE TRUE COST OF POOR QUALITY (CPQ) ---
1. Customer Refunds (Broken items sent back): 2,509,899 EGP
2. Factory Scrap (Items thrown in the trash): 16,322,914 EGP
3. Printing Rework (Ink and fixing mistakes): 26,871,175 EGP
---------------------------------------------
TOTAL MONEY BLEEDING: 45,703,988 EGP


In [ ]:
# Only count ink wasted on rejected + reprinted units, plus rework cost
waste_share = (printing["Rejected_Qty"] + printing["Reprint_Qty"]) / printing["Printed_Qty"]
printing_loss_corrected = printing["Rework_Cost_EGP"].sum() + (printing["Ink_or_Consumable_Cost_EGP"] * waste_share).sum()

total_cpq = total_refund_loss + total_factory_loss + printing_loss_corrected

print("Corrected printing loss:", round(printing_loss_corrected))
print("CORRECTED TOTAL CPQ:", round(total_cpq))

Corrected printing loss: 1954889
CORRECTED TOTAL CPQ: 20787702
